In [ ]:
import os
import json
import shutil
import random
from pathlib import Path
from collections import defaultdict, Counter

import cv2
import numpy as np
from pycocotools import mask as mask_utils


# ============================================================
# USER SETTINGS
# ============================================================

# Official/full COCO annotation files
TRAIN_JSON = (
    "**/D2S_training.json"
)

FULL_VAL_JSON = (
    "**/D2S_validation.json"
)

# Folder containing ALL original D2S images
IMAGE_DIR = (
    "**/images"
)

# Output root
OUTPUT_DIR = (
    "**/all_lights/"
)

# Reproducible validation/test split
RANDOM_SEED = 42

# Fraction of validation groups assigned to validation.
# Remaining groups are used as the held-out test set.
VAL_FRACTION = 0.50


# ============================================================
# LOAD JSON
# ============================================================

def load_json(path):
    with open(path, "r") as f:
        return json.load(f)


# ============================================================
# GET D2S ACQUISITION / SEQUENCE GROUP
# ============================================================

def get_sequence_id(image_id):
    """
    Group related D2S images together.

    Examples:

        800-829   -> sequence 8
        1100-1129 -> sequence 11
        1900-1929 -> sequence 19

    Thus, the different frames / lighting conditions belonging
    to the same acquisition group stay entirely in either
    validation or test.

    This is important to avoid leakage caused by placing closely
    related images in both validation and test.
    """

    return image_id // 100


# ============================================================
# DECODE COCO SEGMENTATION
# ============================================================

def segmentation_to_mask(segmentation, height, width):
    """
    Convert a COCO segmentation into a binary mask.

    Supports:
        - compressed RLE
        - uncompressed RLE
        - polygon COCO annotations
    """

    # --------------------------------------------------------
    # RLE
    # --------------------------------------------------------

    if isinstance(segmentation, dict):

        # Uncompressed RLE
        if isinstance(segmentation.get("counts"), list):

            rle = mask_utils.frPyObjects(
                segmentation,
                height,
                width
            )

        # Compressed RLE
        else:
            rle = segmentation

        mask = mask_utils.decode(rle)

        if mask.ndim == 3:
            mask = np.any(
                mask,
                axis=2
            ).astype(np.uint8)

        return (mask > 0).astype(np.uint8)

    # --------------------------------------------------------
    # COCO polygon
    # --------------------------------------------------------

    elif isinstance(segmentation, list):

        if len(segmentation) == 0:
            return np.zeros(
                (height, width),
                dtype=np.uint8
            )

        rles = mask_utils.frPyObjects(
            segmentation,
            height,
            width
        )

        rle = mask_utils.merge(rles)

        mask = mask_utils.decode(rle)

        if mask.ndim == 3:
            mask = np.any(
                mask,
                axis=2
            ).astype(np.uint8)

        return (mask > 0).astype(np.uint8)

    else:

        raise ValueError(
            f"Unsupported segmentation type: "
            f"{type(segmentation)}"
        )


# ============================================================
# BINARY MASK -> LABELME POLYGONS
# ============================================================

def mask_to_polygons(mask):
    """
    Convert a binary object mask into LabelMe polygons.

    RETR_EXTERNAL:
        extracts external object boundaries.

    CHAIN_APPROX_NONE:
        keeps every contour point.

    No polygon simplification is performed.
    """

    mask = (mask > 0).astype(np.uint8)

    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_NONE
    )

    polygons = []

    for contour in contours:

        # Need at least 3 points for a polygon
        if len(contour) < 3:
            continue

        contour = contour.reshape(-1, 2)

        points = [
            [float(x), float(y)]
            for x, y in contour
        ]

        polygons.append(points)

    return polygons


# ============================================================
# CREATE LABELME JSON
# ============================================================

def create_labelme_json(
    image_info,
    annotations,
    category_names
):
    """
    Convert all COCO annotations belonging to one image into
    one LabelMe JSON dictionary.
    """

    width = image_info["width"]
    height = image_info["height"]

    filename = os.path.basename(
        image_info["file_name"]
    )

    shapes = []

    for ann in annotations:

        segmentation = ann.get("segmentation")

        if segmentation is None:
            continue

        category_id = ann["category_id"]

        if category_id not in category_names:
            print(
                f"WARNING: unknown category ID "
                f"{category_id}"
            )
            continue

        label = category_names[category_id]

        # ----------------------------------------------------
        # Decode COCO segmentation
        # ----------------------------------------------------

        mask = segmentation_to_mask(
            segmentation,
            height,
            width
        )

        # Empty mask
        if not np.any(mask):
            continue

        # ----------------------------------------------------
        # Binary mask -> polygon(s)
        # ----------------------------------------------------

        polygons = mask_to_polygons(mask)

        for polygon in polygons:

            shapes.append(
                {
                    "label": label,
                    "points": polygon,

                    # Disconnected components belonging to the
                    # same COCO instance receive the same ID.
                    "group_id": int(ann["id"]),

                    "description": "",
                    "shape_type": "polygon",
                    "flags": {},
                    "mask": None
                }
            )

    return {
        "version": "5.5.0",
        "flags": {},
        "shapes": shapes,
        "imagePath": filename,
        "imageData": None,
        "imageHeight": height,
        "imageWidth": width
    }


# ============================================================
# BUILD ANNOTATION LOOKUP
# ============================================================

def build_annotation_lookup(coco):
    """
    Build:
        image_id -> list of COCO annotations
    """

    annotations_by_image = defaultdict(list)

    for ann in coco["annotations"]:

        annotations_by_image[
            ann["image_id"]
        ].append(ann)

    return annotations_by_image


# ============================================================
# COPY + CONVERT ONE DATASET SPLIT
# ============================================================

def process_split(
    coco,
    selected_image_ids,
    split_name
):
    """
    Copy selected images and convert their COCO annotations
    into individual LabelMe JSON files.
    """

    print()
    print("=" * 70)
    print(f"PROCESSING {split_name.upper()}")
    print("=" * 70)

    output_image_dir = os.path.join(
        OUTPUT_DIR,
        split_name,
        "images"
    )

    output_json_dir = os.path.join(
        OUTPUT_DIR,
        split_name,
        "jsons"
    )

    os.makedirs(
        output_image_dir,
        exist_ok=True
    )

    os.makedirs(
        output_json_dir,
        exist_ok=True
    )

    # --------------------------------------------------------
    # Category mapping
    # --------------------------------------------------------

    category_names = {
        cat["id"]: cat["name"]
        for cat in coco["categories"]
    }

    # --------------------------------------------------------
    # Image mapping
    # --------------------------------------------------------

    images = {
        img["id"]: img
        for img in coco["images"]
    }

    # --------------------------------------------------------
    # Annotation mapping
    # --------------------------------------------------------

    annotations_by_image = (
        build_annotation_lookup(coco)
    )

    converted_images = 0
    converted_instances = 0
    missing_images = 0

    selected_image_ids = sorted(
        selected_image_ids
    )

    for image_id in selected_image_ids:

        image_info = images.get(image_id)

        if image_info is None:

            print(
                f"WARNING: image ID {image_id} "
                f"not found in COCO JSON."
            )

            continue

        filename = image_info["file_name"]

        # ----------------------------------------------------
        # Locate original image
        # ----------------------------------------------------

        source_path = os.path.join(
            IMAGE_DIR,
            filename
        )

        # In case file_name contains a subdirectory
        if not os.path.exists(source_path):

            source_path = os.path.join(
                IMAGE_DIR,
                os.path.basename(filename)
            )

        if not os.path.exists(source_path):

            print(
                f"WARNING: image not found: "
                f"{filename}"
            )

            missing_images += 1
            continue

        output_filename = os.path.basename(
            filename
        )

        # ----------------------------------------------------
        # Copy image
        # ----------------------------------------------------

        destination_path = os.path.join(
            output_image_dir,
            output_filename
        )

        shutil.copy2(
            source_path,
            destination_path
        )

        # ----------------------------------------------------
        # Convert annotations
        # ----------------------------------------------------

        annotations = annotations_by_image.get(
            image_id,
            []
        )

        labelme = create_labelme_json(
            image_info,
            annotations,
            category_names
        )

        json_filename = (
            Path(output_filename).stem
            + ".json"
        )

        json_path = os.path.join(
            output_json_dir,
            json_filename
        )

        with open(json_path, "w") as f:

            json.dump(
                labelme,
                f,
                indent=2
            )

        converted_images += 1
        converted_instances += len(annotations)

    print(
        f"{split_name} images converted: "
        f"{converted_images}"
    )

    print(
        f"{split_name} COCO instances: "
        f"{converted_instances}"
    )

    print(
        f"{split_name} missing image files: "
        f"{missing_images}"
    )


# ============================================================
# GET CLASS DISTRIBUTION
# ============================================================

def get_class_distribution(
    coco,
    image_ids
):
    """
    Count COCO instances of every category for a set of images.
    """

    image_ids = set(image_ids)

    distribution = Counter()

    for ann in coco["annotations"]:

        if ann["image_id"] in image_ids:

            distribution[
                ann["category_id"]
            ] += 1

    return distribution


# ============================================================
# SPLIT FULL OFFICIAL VALIDATION SET
# ============================================================

def split_validation_by_sequence(val_coco):
    """
    Split the COMPLETE official D2S validation set into:

        validation
        held-out test

    IMPORTANT:
    The split is performed at sequence/acquisition-group level,
    NOT at individual-image level.

    Therefore all images belonging to the same sequence remain
    together.

    No sequence is allowed to occur in both validation and test.
    """

    all_images = val_coco["images"]

    print()
    print("=" * 70)
    print("FULL VALIDATION DATASET")
    print("=" * 70)

    print(
        "Total official validation images:",
        len(all_images)
    )

    # --------------------------------------------------------
    # Group ALL validation images by sequence
    # --------------------------------------------------------

    groups = defaultdict(list)

    for img in all_images:

        sequence_id = get_sequence_id(
            img["id"]
        )

        groups[
            sequence_id
        ].append(
            img["id"]
        )

    # Sort images within each group
    for sequence_id in groups:

        groups[sequence_id] = sorted(
            groups[sequence_id]
        )

    sequence_ids = sorted(
        groups.keys()
    )

    print(
        "Number of acquisition groups:",
        len(sequence_ids)
    )

    # --------------------------------------------------------
    # Show group-size statistics
    # --------------------------------------------------------

    group_sizes = [
        len(groups[sequence_id])
        for sequence_id in sequence_ids
    ]

    print(
        "Minimum group size:",
        min(group_sizes)
    )

    print(
        "Maximum group size:",
        max(group_sizes)
    )

    print(
        "Mean group size:",
        sum(group_sizes) / len(group_sizes)
    )

    # --------------------------------------------------------
    # Reproducibly shuffle COMPLETE groups
    # --------------------------------------------------------

    rng = random.Random(
        RANDOM_SEED
    )

    shuffled_sequences = (
        sequence_ids.copy()
    )

    rng.shuffle(
        shuffled_sequences
    )

    # --------------------------------------------------------
    # Assign complete groups while targeting approximately
    # VAL_FRACTION of the images for validation.
    #
    # This is better than simply splitting the number of
    # groups if some groups have different numbers of images.
    # --------------------------------------------------------

    total_images = len(all_images)

    target_val_images = round(
        total_images * VAL_FRACTION
    )

    val_sequences = set()
    test_sequences = set()

    val_count = 0

    for sequence_id in shuffled_sequences:

        group_size = len(
            groups[sequence_id]
        )

        # Put groups into validation until approximately
        # the target number of validation images is reached.
        if val_count < target_val_images:

            val_sequences.add(
                sequence_id
            )

            val_count += group_size

        else:

            test_sequences.add(
                sequence_id
            )

    # --------------------------------------------------------
    # Convert group IDs -> image IDs
    # --------------------------------------------------------

    val_ids = set()

    for sequence_id in val_sequences:

        val_ids.update(
            groups[sequence_id]
        )

    test_ids = set()

    for sequence_id in test_sequences:

        test_ids.update(
            groups[sequence_id]
        )

    # ========================================================
    # STRICT LEAKAGE CHECKS
    # ========================================================

    assert val_sequences.isdisjoint(
        test_sequences
    ), (
        "SEQUENCE/GROUP LEAKAGE DETECTED!"
    )

    assert val_ids.isdisjoint(
        test_ids
    ), (
        "IMAGE LEAKAGE DETECTED!"
    )

    all_validation_ids = {
        img["id"]
        for img in all_images
    }

    assert (
        val_ids | test_ids
    ) == all_validation_ids, (
        "Some validation images were lost!"
    )

    assert (
        len(val_ids) + len(test_ids)
        == len(all_validation_ids)
    ), (
        "Duplicate/lost images detected!"
    )

    # --------------------------------------------------------
    # Check sequence assignment directly for every image
    # --------------------------------------------------------

    for image_id in val_ids:

        sequence_id = get_sequence_id(
            image_id
        )

        assert (
            sequence_id not in test_sequences
        ), (
            f"Leakage: sequence {sequence_id} "
            f"occurs in both VAL and TEST."
        )

    for image_id in test_ids:

        sequence_id = get_sequence_id(
            image_id
        )

        assert (
            sequence_id not in val_sequences
        ), (
            f"Leakage: sequence {sequence_id} "
            f"occurs in both TEST and VAL."
        )

    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    print()
    print("=" * 70)
    print("VAL / TEST SPLIT")
    print("=" * 70)

    print(
        "VAL groups:",
        len(val_sequences)
    )

    print(
        "TEST groups:",
        len(test_sequences)
    )

    print(
        "VAL images:",
        len(val_ids)
    )

    print(
        "TEST images:",
        len(test_ids)
    )

    print(
        "Total:",
        len(val_ids) + len(test_ids)
    )

    print()
    print(
        "VAL group IDs:"
    )

    print(
        sorted(val_sequences)
    )

    print()
    print(
        "TEST group IDs:"
    )

    print(
        sorted(test_sequences)
    )

    print()
    print(
        "✓ No image overlap"
    )

    print(
        "✓ No acquisition-group overlap"
    )

    print(
        "✓ Every official validation image "
        "is assigned exactly once"
    )

    return (
        val_ids,
        test_ids,
        val_sequences,
        test_sequences
    )


# ============================================================
# PRINT CLASS DISTRIBUTION
# ============================================================

def print_class_distribution(
    coco,
    val_ids,
    test_ids
):
    """
    Print per-category instance counts for validation and test.
    This does not alter the split.
    """

    category_names = {
        cat["id"]: cat["name"]
        for cat in coco["categories"]
    }

    val_distribution = (
        get_class_distribution(
            coco,
            val_ids
        )
    )

    test_distribution = (
        get_class_distribution(
            coco,
            test_ids
        )
    )

    print()
    print("=" * 70)
    print("CLASS DISTRIBUTION")
    print("=" * 70)

    print(
        f"{'Class':40s} "
        f"{'VAL':>8s} "
        f"{'TEST':>8s}"
    )

    print("-" * 60)

    for category_id in sorted(
        category_names.keys()
    ):

        name = category_names[
            category_id
        ]

        val_count = val_distribution[
            category_id
        ]

        test_count = test_distribution[
            category_id
        ]

        print(
            f"{name:40s} "
            f"{val_count:8d} "
            f"{test_count:8d}"
        )


# ============================================================
# SAVE SPLIT INFORMATION
# ============================================================

def save_split_information(
    train_ids,
    val_ids,
    test_ids,
    val_sequences,
    test_sequences
):
    """
    Save exact image/group assignments so that the same split
    can always be reproduced and documented in the paper.
    """

    split_information = {

        "description":
            "Full D2S training set and sequence-level "
            "50/50 split of the official validation set.",

        "random_seed":
            RANDOM_SEED,

        "validation_fraction":
            VAL_FRACTION,

        "train_image_count":
            len(train_ids),

        "val_image_count":
            len(val_ids),

        "test_image_count":
            len(test_ids),

        "train_images":
            sorted(train_ids),

        "val_images":
            sorted(val_ids),

        "test_images":
            sorted(test_ids),

        "val_sequences":
            sorted(val_sequences),

        "test_sequences":
            sorted(test_sequences)
    }

    os.makedirs(
        OUTPUT_DIR,
        exist_ok=True
    )

    output_path = os.path.join(
        OUTPUT_DIR,
        "split_information.json"
    )

    with open(
        output_path,
        "w"
    ) as f:

        json.dump(
            split_information,
            f,
            indent=2
        )

    print()
    print(
        "Saved exact split information to:"
    )

    print(
        output_path
    )


# ============================================================
# MAIN
# ============================================================

def main():

    # --------------------------------------------------------
    # Load annotations
    # --------------------------------------------------------

    print(
        "Loading FULL training annotations..."
    )

    train_coco = load_json(
        TRAIN_JSON
    )

    print(
        "Loading FULL validation annotations..."
    )

    val_coco = load_json(
        FULL_VAL_JSON
    )

    # ========================================================
    # FULL TRAINING SET
    # ========================================================

    train_ids = {
        img["id"]
        for img in train_coco["images"]
    }

    print()
    print("=" * 70)
    print("TRAINING DATASET")
    print("=" * 70)

    print(
        "Training images:",
        len(train_ids)
    )

    print(
        "Training annotations:",
        len(train_coco["annotations"])
    )

    # ========================================================
    # SPLIT FULL VALIDATION -> VAL + TEST
    # ========================================================

    (
        val_ids,
        test_ids,
        val_sequences,
        test_sequences

    ) = split_validation_by_sequence(
        val_coco
    )

    # ========================================================
    # FINAL TRAIN / VAL / TEST LEAKAGE CHECK
    # ========================================================

    assert train_ids.isdisjoint(
        val_ids
    ), (
        "TRAIN / VAL image-ID overlap found!"
    )

    assert train_ids.isdisjoint(
        test_ids
    ), (
        "TRAIN / TEST image-ID overlap found!"
    )

    assert val_ids.isdisjoint(
        test_ids
    ), (
        "VAL / TEST image-ID overlap found!"
    )

    print()
    print("=" * 70)
    print("FINAL LEAKAGE CHECK")
    print("=" * 70)

    print(
        "✓ Train and val image IDs do not overlap"
    )

    print(
        "✓ Train and test image IDs do not overlap"
    )

    print(
        "✓ Val and test image IDs do not overlap"
    )

    print(
        "✓ Val/test acquisition groups do not overlap"
    )

    # ========================================================
    # CLASS DISTRIBUTION
    # ========================================================

    print_class_distribution(
        val_coco,
        val_ids,
        test_ids
    )

    # ========================================================
    # PROCESS TRAIN
    # ========================================================

    process_split(
        coco=train_coco,
        selected_image_ids=train_ids,
        split_name="train"
    )

    # ========================================================
    # PROCESS VAL
    # ========================================================

    process_split(
        coco=val_coco,
        selected_image_ids=val_ids,
        split_name="val"
    )

    # ========================================================
    # PROCESS TEST
    # ========================================================

    process_split(
        coco=val_coco,
        selected_image_ids=test_ids,
        split_name="test"
    )

    # ========================================================
    # SAVE EXACT SPLIT
    # ========================================================

    save_split_information(
        train_ids,
        val_ids,
        test_ids,
        val_sequences,
        test_sequences
    )

    # ========================================================
    # FINAL SUMMARY
    # ========================================================

    print()
    print("=" * 70)
    print("FINAL DATASET")
    print("=" * 70)

    print(
        f"Train: {len(train_ids)} images"
    )

    print(
        f"Val:   {len(val_ids)} images"
    )

    print(
        f"Test:  {len(test_ids)} images"
    )

    print(
        f"Total: "
        f"{len(train_ids) + len(val_ids) + len(test_ids)} "
        f"images"
    )

    print()
    print(
        "Output directory:"
    )

    print(
        OUTPUT_DIR
    )

    print()
    print(
        "Done."
    )


if __name__ == "__main__":
    main()

Loading FULL training annotations...
Loading FULL validation annotations...

TRAINING DATASET
Training images: 4380
Training annotations: 6900

FULL VALIDATION DATASET
Total official validation images: 3600
Number of acquisition groups: 120
Minimum group size: 30
Maximum group size: 30
Mean group size: 30.0

VAL / TEST SPLIT
VAL groups: 60
TEST groups: 60
VAL images: 1800
TEST images: 1800
Total: 3600

VAL group IDs:
[11, 19, 38, 48, 66, 89, 135, 225, 233, 235, 242, 251, 253, 260, 272, 279, 288, 300, 302, 309, 315, 334, 341, 351, 354, 368, 382, 392, 399, 414, 432, 434, 437, 443, 447, 451, 467, 477, 505, 516, 518, 533, 542, 552, 560, 565, 567, 569, 573, 583, 586, 587, 601, 619, 641, 662, 666, 670, 681, 697]

TEST group IDs:
[8, 27, 32, 33, 92, 95, 105, 108, 112, 131, 162, 227, 230, 252, 257, 258, 259, 270, 277, 280, 281, 286, 316, 318, 327, 333, 335, 374, 381, 397, 398, 440, 471, 480, 486, 493, 501, 504, 508, 541, 548, 553, 559, 566, 568, 578, 627, 631, 634, 645, 648, 652, 653, 667, 668